In [14]:
import numpy as np
import scipy.linalg as la
import torch.nn as nn

In [15]:
from qqe.GNN.training.utils import collect_files_path, cache_root_paths
from qqe.GNN.training.datasets import build_loaders_NN
from qqe.GNN.physics_aware_NN import GNN, NN, GlobalMLP
from qqe.GNN.training.train import build_loss, train_model
from qqe.GNN.training.train_config import TrainConfig

In [16]:
epochs=30
lr=0.001
loss_type="mse"
training_mode="per_family"
family="random"
target="sre"
show_progress=True
show_val_progress=True
log_every_n_batches=5
heartbeat_secs=60.0
epoch_time_warning_secs=300.0

In [4]:
train_config = TrainConfig(
        epochs=epochs,
        lr=lr,
        loss_type=loss_type,
        training_mode=training_mode,
        family=family,
        target=target,
        show_progress=show_progress,
        show_val_progress=show_val_progress,
        log_batch_loss_every=log_every_n_batches,
        heartbeat=heartbeat_secs,
        epoch_warning=epoch_time_warning_secs,
    )

In [5]:
family_filter = train_config.family if train_config.training_mode == "per_family" else None
family_projection = train_config.family if train_config.training_mode == "per_family" else None

In [6]:
data_paths = collect_files_path("../outputs/data", family=family_filter)

In [7]:
print(f"Found {len(data_paths)} data files for training.")

Found 50000 data files for training.


In [8]:
train_loader, val_loader, test_loader, global_in_dim, base_dataset = build_loaders_NN(
        data_paths,
        batch_size=train_config.batch_size,
        seed=train_config.seed,
        train_split=train_config.train_split,
        val_split=train_config.val_split,
        global_feature_variant=train_config.global_feature_variant,
        node_feature_variant=train_config.node_feature_backend_variant,
        family_projection=family_projection,
    )

In [9]:
global_in_dim

153

In [10]:
print(base_dataset[0])

Data(
  x=[1348, 73],
  edge_index=[2, 1666],
  y=[1],
  global_features=[1, 153],
  num_qubits=10,
  gate_counts={
    CNOT_count=328,
    rx_bin_0=8,
    rx_bin_1=5,
    rx_bin_10=7,
    rx_bin_11=8,
    rx_bin_12=8,
    rx_bin_13=4,
    rx_bin_14=10,
    rx_bin_15=10,
    rx_bin_16=8,
    rx_bin_17=3,
    rx_bin_18=3,
    rx_bin_19=7,
    rx_bin_2=12,
    rx_bin_20=10,
    rx_bin_21=4,
    rx_bin_22=9,
    rx_bin_23=7,
    rx_bin_24=8,
    rx_bin_25=4,
    rx_bin_26=9,
    rx_bin_27=6,
    rx_bin_28=6,
    rx_bin_29=8,
    rx_bin_3=6,
    rx_bin_30=10,
    rx_bin_31=6,
    rx_bin_32=6,
    rx_bin_33=8,
    rx_bin_34=3,
    rx_bin_35=7,
    rx_bin_36=8,
    rx_bin_37=3,
    rx_bin_38=5,
    rx_bin_39=7,
    rx_bin_4=10,
    rx_bin_40=10,
    rx_bin_41=10,
    rx_bin_42=6,
    rx_bin_43=5,
    rx_bin_44=9,
    rx_bin_45=5,
    rx_bin_46=5,
    rx_bin_47=6,
    rx_bin_48=4,
    rx_bin_49=3,
    rx_bin_5=7,
    rx_bin_6=5,
    rx_bin_7=8,
    rx_bin_8=4,
    rx_bin_9=7,
    ry_bin_0=8,


In [17]:
model_NN = NN(
    global_in_dim = global_in_dim,
    global_hidden = 64,
    reg_hidden = 128,
    dropout_rate = 0.1,
)

model_MLP = GlobalMLP(
    in_dim=global_in_dim,
    hidden_dim=64,
    dropout_rate=0.1,
)

model_tabular = nn.Sequential(
    GlobalMLP(in_dim=global_in_dim, hidden_dim=64, dropout_rate=0.1),
    nn.Linear(64, 1),
)

In [ ]:
model, hist, dev = train_model(
    model_NN,
    train_loader,
    val_loader,
    epochs=train_config.epochs,
    lr=train_config.lr,
    loss_type=train_config.loss_type,
    scheduler="plateau",
    show_progress=train_config.show_progress,
    show_val_progress=train_config.show_val_progress,
    log_every_n_batches=train_config.log_batch_loss_every,
    heartbeat_secs=train_config.heartbeat,
    epoch_time_warning_secs=train_config.epoch_warning,
)

Using device: cuda


KeyboardInterrupt: 

In [68]:
for s in range(50):
    rng = np.random.default_rng(s)
    X = haar_unitary_gate(2 ** 2, rng)
    counts = eigenvalues_bins(X)
    print(f"{s}: {counts}")

0: [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 1 0 0 0 0 0 0 0 1 0 0 0 0]
1: [0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0
 0 0 0 1 0 0 0 0 0 0 0 0 0]
2: [0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 1 0 0 0 0 0 0 0 0]
3: [0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0]
4: [0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0]
5: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0
 1 0 0 0 0 1 0 0 0 0 0 0 0]
6: [0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0
 0 0 0 0 0 0 0 0 0 0 0 0 0]
7: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1
 0 0 0 0 0 0 0 1 0 0 0 0 0]
8: [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0
 0 0 0 0 0 0 1 0 0 0 0 0 0]
9: [0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 